# Comparing Sentiment Analysis Methods on Social Media Text
**Cameron Batts**

Python · NLP · Naive Bayes · Logistic Regression · VADER · TextBlob

## Project overview
Compare supervised text classifiers with lexicon-based sentiment scoring using public NLTK Twitter samples. The analysis explores model behavior, vocabulary coverage, and sensitivity to emotionally loaded words.

The proposed business application is radiology patient-feedback analysis. This is a methods demonstration on public tweets, not a deployment or evaluation on patient data.

Selected aggregate outputs are retained from the supplied analysis; text examples containing third-party names and handles are not included as saved outputs. Rerunning the notebook loads the original public corpus and can display those names. The workflow has not been re-executed during portfolio preparation.


## Validation update - October 22, 2026
The full 14-cell workflow reproduced 20,000 sample tweets, mean length 121.5088 characters and 20,770 vocabulary terms. Emoticon matching found 482 positive and 14 negative matches: 496 unique tweets and 0 in both categories. The controlled sentence change moved Naive Bayes positive probability from 0.2220 to 0.9044. No labeled holdout or healthcare outcome evaluation was performed.


## Business context
Patient feedback about check-in, wait times, staff communication, and exam experience could help healthcare leaders identify service concerns. Potential measures include patient satisfaction, complaint trends, wait times, and turnaround times. These are proposed uses and measures, not demonstrated improvements.

Sarcasm, mixed sentiment, and domain-specific language require careful interpretation and human review.

## Data and environment
The unlabeled analysis corpus contains 20,000 tweets. Separate positive and negative corpora train the classifiers. Install dependencies from `requirements.txt` before running. NLTK downloads require internet access.

Data source: NLTK `twitter_samples` (https://www.nltk.org/nltk_data/). Tweet content is loaded at runtime rather than bundled. Preserve applicable corpus terms and attribution.

In [1]:
import pandas as pd
import nltk

def ensure_nltk_resource(location, package):
    """Reuse local resources; download only when absent."""
    try:
        nltk.data.find(location)
    except LookupError:
        if not nltk.download(package, quiet=True):
            raise RuntimeError(f'Install the NLTK {package} resource locally, then rerun.')

from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


In [2]:
# Public NLTK Twitter samples; no patient data is used.
ensure_nltk_resource('corpora/twitter_samples', 'twitter_samples')
from nltk.corpus import twitter_samples
sample_df = pd.DataFrame({'text': twitter_samples.strings('tweets.20150430-223406.json')})
positive_df = pd.DataFrame({'text': twitter_samples.strings('positive_tweets.json')})
negative_df = pd.DataFrame({'text': twitter_samples.strings('negative_tweets.json')})


## Text length profile
The original run reported an average of 121.51 characters across 20,000 tweets.

In [3]:
sample_df['text_length'] = sample_df['text'].apply(len)
print(sample_df['text_length'].describe())

count    20000.000000
mean       121.508800
std         27.536847
min          4.000000
25%        111.000000
50%        137.000000
75%        140.000000
max        152.000000
Name: text_length, dtype: float64


## Naive Bayes baseline
Fit a bag-of-words classifier on the labeled corpora and score the separate unlabeled corpus. Positive-class probabilities are model estimates, not measured correctness.

In [4]:
positive_df['y'] = 1
negative_df['y'] = 0
labeled_df = pd.concat([positive_df, negative_df])


X = labeled_df['text']
y = labeled_df['y']
vectorizer = CountVectorizer()
X_v = vectorizer.fit_transform(X)
model_nb = MultinomialNB()
model_nb.fit(X_v, y)

In [5]:
X_sample = sample_df['text']
X_sample_v = vectorizer.transform(X_sample)

# Compute probabilities using the NB model, on the sample tweets - this returns (gives us) a array (20000x2) for 20,000 tweets for my two classes (negative and positive)
y_sample_pred_probs = model_nb.predict_proba(X_sample_v)

# To make it easier to manipulate the data, we are going to convert that array (list) to a DataFrame which makes it easier to work with.
probs_df = pd.DataFrame(y_sample_pred_probs)

#Attach/copy the predicted probability for each of the tweet being positive
sample_df['NBPosScore'] = probs_df[1]

#Drop all columns except the tweets and NBPosScore
sample_df = sample_df[['text','NBPosScore']]

In [6]:
print(f"Vocabulary Size: {len(vectorizer.get_feature_names_out())}")

Vocabulary Size: 20770


## Qualitative ranking review
The original inspection found that high positive scores could occur for spam-like or sarcastic text. Bag-of-words Naive Bayes does not explicitly model word order or context, and its conditional independence assumption limits interpretation. The 20,770-feature vocabulary reflects the training corpus and vectorizer settings.

No ground-truth labels are supplied for the 20,000-tweet analysis corpus, so this notebook does not establish accuracy or superiority over other methods.

## Prebuilt analyzers
TextBlob polarity and VADER compound scores range from −1 to 1; they are not on the same scale as classifier probabilities. Their sensitivity to positive words does not establish better accuracy.

In [7]:
# SentimentIntensityAnalyzer is a VADER sentiment analysis tool from nltk
from nltk.sentiment import SentimentIntensityAnalyzer
ensure_nltk_resource('sentiment/vader_lexicon.zip', 'vader_lexicon')

# TextBlob provides a simple API for common NLP tasks
from textblob import TextBlob

In [8]:
# TextBlob scores
sample_df['TBScore'] = sample_df['text'].apply(lambda x: TextBlob(x).sentiment.polarity)

# SIA scores
sia = SentimentIntensityAnalyzer()
sample_df['SIAScore'] = sample_df['text'].apply(lambda x: sia.polarity_scores(x)['compound'])



## Emoticon coverage
An explicit emoticon lexicon provides a transparent, narrow baseline. Matching is substring-based and can produce false positives.

In [9]:
#positive_lexicons
positive_lexicons = [ ':-)', ':)', ';)', ':o)', ':]', ':3', ':c)', ':>', '=]', '8)', '=)', ':}',
    ':^)', ':-D', ':D', '8-D', '8D', 'x-D', 'xD', 'X-D', 'XD', '=-D', '=D',
    '=-3', '=3', ':-))', ":'-)", ":')", ':*', ':^*', '>:P', ':-P', ':P', 'X-P',
    'x-p', 'xp', 'XP', ':-p', ':p', '=p', ':-b', ':b', '>:)', '>;)', '>:-)',
    '<3']

negative_lexicons = [':L', ':-/', '>:/', ':S', '>:[', ':@', ':-(', ':[', ':-||', '=L', ':<',
    ':-[', ':-<', '=\\', '=/', '>:(', ':(', '>.<', ":'-(", ":'(", ':\\', ':-c',
    ':c', ':{', '>:\\', ';(']


In [10]:
pos_count = sample_df['text'].apply(lambda x: any(lex in x for lex in positive_lexicons)).sum()
neg_count = sample_df['text'].apply(lambda x: any(lex in x for lex in negative_lexicons)).sum()

print("Tweets with positive lexicons:", pos_count)
print("Tweets with negative lexicons:", neg_count)

Tweets with positive lexicons: 482
Tweets with negative lexicons: 14


The original run found 482 tweets with positive emoticons and 14 with negative emoticons. These counts may overlap; 496 is not a confirmed count of unique covered tweets. Coverage and accuracy are different measures.

In [11]:
pos_mask = sample_df['text'].apply(lambda text: any(term in text for term in positive_lexicons))
neg_mask = sample_df['text'].apply(lambda text: any(term in text for term in negative_lexicons))
print("Unique tweets matched:", int((pos_mask | neg_mask).sum()))
print("Both categories:", int((pos_mask & neg_mask).sum()))


Unique tweets matched: 496
Both categories: 0


## Logistic regression comparison
Fit a second supervised classifier on the same bag-of-words representation. A fair predictive comparison would require a held-out labeled set.

In [12]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_v, y)

lr_probs = lr_model.predict_proba(X_sample_v)
sample_df['LRPosScore'] = lr_probs[:, 1]



## Sensitivity to positive-token injection
Compare a negative sentence with a modified version that adds positive tokens. This is an illustrative sensitivity check, not a comprehensive robustness benchmark.

In [13]:
sentence = "The Thai food in this town tastes terrible."

print("SIA:", sia.polarity_scores(sentence))
print("TextBlob:", TextBlob(sentence).sentiment.polarity)
print("Naive Bayes:", model_nb.predict_proba(vectorizer.transform([sentence]))[0][1])

SIA: {'neg': 0.307, 'neu': 0.693, 'pos': 0.0, 'compound': -0.4767}
TextBlob: -1.0
Naive Bayes: 0.22200168062658243


In [14]:
new_sentence = "love good great The Thai food in this town tastes terrible :)"

print("New sentence:", new_sentence)
print("SIA:", sia.polarity_scores(new_sentence))
print("TextBlob:", TextBlob(new_sentence).sentiment.polarity)
print("Naive Bayes:", model_nb.predict_proba(vectorizer.transform([new_sentence]))[0][1])

New sentence: love good great The Thai food in this town tastes terrible :)
SIA: {'neg': 0.128, 'neu': 0.288, 'pos': 0.584, 'compound': 0.9022}
TextBlob: 0.3
Naive Bayes: 0.9043949417063923


## Findings and limitations
- The baseline sentence scored −0.4767 in VADER, −1.0 in TextBlob, and approximately 0.222 positive probability in Naive Bayes.
- Adding positive words and an emoticon shifted the respective outputs to 0.9022, 0.3, and approximately 0.904. Sentiment estimates can be strongly affected by surface wording.
- Emoticon matching has limited coverage, while full-text methods score every document; neither observation proves correctness.
- No held-out accuracy, calibration, or patient-domain performance was measured. Duplicate posts may influence aggregate patterns.

## Next steps
Create a labeled evaluation set, control duplicate leakage across splits, report precision/recall/F1, and evaluate negation, sarcasm, and mixed sentiment. Validate separately on appropriately governed patient-feedback data before operational use.
